<a href="https://colab.research.google.com/github/JakeEisner/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2012/%5BLab_12_%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Lab Objective: Transitioning from Explanation to Prediction
Build a multivariate Ordinary Least Squares (OLS) model using the statsmodels formula API, shifting the analytical focus from causal parameter estimation to minimizing out-of-sample predictive loss. You will utilize contemporary 2026 digital real estate data to calculate and interpret the Root Mean Squared Error (RMSE) in real-world units (US Dollars).
⚠️ The Observational Challenge
In the dynamic technology sector, assuming environments are static is a critical error. Classical econometric models often fail to account for structural breaks, regime changes, and highly localized price elasticity. Furthermore, applying an algorithmic prediction to adjust pricing in a dynamic marketplace can fundamentally alter human behavior, leading to substitution effects and spillover bias (a violation of the Stable Unit Treatment Value Assumption, or SUTVA). You must be highly vigilant: a model with a high R-squared but a massive RMSE will lead to catastrophic financial miscalculations, such as acquiring toxic real estate inventory at inflated prices.

In [10]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/JakeEisner/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [11]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age +	Distance_to_Transit +	School_District_Rating'

In [12]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        20:05:19   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [13]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [14]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)

# Print the result formatted as a $USD currency string
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


🤖 AI-Assisted Expansion
The Generative AI Policy: Foundations First, Expansion Second. Now that you have established manual mastery over the syntax and understand the core mathematics of the OLS prediction engine, you are authorized to operate under the "Co-Pilot Rule." Use the P.R.I.M.E. Framework to scale your work into visual diagnostics.

Your Expansion Task: Build an Interactive Residual Forensics Dashboard

In [15]:
# Step 6: Interactive Residual Forensics Dashboard
import plotly.express as px
import numpy as np

# Extract fitted (predicted) values from the statsmodels results object
# results.fittedvalues returns the model's in-sample predicted y-values
fitted_vals = results.fittedvalues

# Extract residuals from the statsmodels results object
# results.resid returns the error for each observation:
# actual value - predicted value
residuals = results.resid

# Compute the residual standard deviation so we can flag unusual observations
resid_std = residuals.std()

# Define outliers as observations with residuals more than 2 standard deviations
# away from zero
outlier_mask = np.abs(residuals) > 2 * resid_std

# Build a clean plotting DataFrame
plot_df = df.copy()
plot_df["Fitted_Values"] = fitted_vals
plot_df["Residuals"] = residuals
plot_df["Outlier_Flag"] = np.where(outlier_mask, "Outlier (> 2 SD)", "Normal")

# Create interactive scatter plot
fig = px.scatter(
    plot_df,
    x="Fitted_Values",
    y="Residuals",
    color="Outlier_Flag",
    color_discrete_map={
        "Normal": "steelblue",
        "Outlier (> 2 SD)": "crimson"
    },
    hover_data=plot_df.columns,
    title="Interactive Residual Forensics Dashboard: Fitted Values vs Residuals",
    labels={
        "Fitted_Values": "Fitted Predicted Values",
        "Residuals": "Residual Errors"
    }
)

# Add horizontal zero-line so residual spread is easy to judge
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    annotation_text="Zero Residual Line",
    annotation_position="top left"
)

# Improve layout for readability
fig.update_traces(marker=dict(size=9, opacity=0.8))
fig.update_layout(
    template="plotly_white",
    height=600,
    width=950
)

fig.show()